In [1]:
# notebook for getting nwm data from server and making nc file
import sys
import os
import subprocess
import shutil
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import gsw
import pickle
import glob
import urllib.request
from urllib.error import URLError, HTTPError


from datetime import datetime, timedelta, date

sys.path.append('../sdpm_py_util')
import init_funs_forecast as initfuns
import run_funs as runfuns
import util_functions as utilfuns
import river_functions as riverfuns
import ocn_funs_forecast as ocnfuns
import swan_functions as swanfuns
import run_funs as runfuns
sys.path.append('/home/mspydell/models/PFM_root/PFM/driver')
import driver_functions as driverfuns
#from driver_run_forecast_LV1234 import driver_run_fore_LV1234
from driver_run_pfm_phm import driver_run_pfm_phm



In [2]:
def make_Qnc(t2,Q,rids,file_name_out):
    # makes an nc file of discharge data for multiple rivers
    # t2 is the time stamps of the flow as an array of datetimes
    # Q is the nt by n_river np array of discharge
    # rids are the reach ids of the data
    # riv_name is a np array of strings that name each river.

    #reach_ids = [948070199, 20331702, 20324441]
    # the order of reach_ids and names should correspond.
    station_ids = np.array(['Sweewater','Otay','TJRE'])
    num_stations = len(station_ids)


    with nc.Dataset(file_name_out, 'w', format='NETCDF4') as nc_file:
        # Create dimensions
        nc_file.createDimension('time', None)  # Unlimited dimension for time
        nc_file.createDimension('station', num_stations)

        # Create variables
        time_var = nc_file.createVariable('time', 'f8', ('time',))
        q_var = nc_file.createVariable('discharge', 'f8', ('time', 'station'))
        station_id_var = nc_file.createVariable('station_id', str, ('station',)) # For string station IDs
        reach_id_var = nc_file.createVariable('reach_id', int, ('station',)) # For string station IDs

        # Add attributes to variables (optional, but recommended for CF-compliance)
        time_var.units = 'days since 1999-01-01 00:00:00'
        time_var.calendar = 'gregorian'
        q_var.units = 'm^3/s'
        q_var.long_name = 'river discharge'
        station_id_var.long_name = 'river name'
        reach_id_var.long_name = 'reach ids from National Water Model'

        # Add global attributes (optional)
        nc_file.title = 'discharge from National Water Model: medium range blend forecast'
        nc_file.history = f'Created on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'

        # 3. Write data to variables
        time_var[:] = nc.date2num(t2, units=time_var.units, calendar=time_var.calendar)
        q_var[:] = Q
        station_id_var[:] = station_ids
        reach_id_var[:] = rids




In [3]:
reach_ids = [948070199, 20331702, 20324441]
            #SW         Otay      TJR 20324441 is last segment near ocean.
reach_ids = np.array(reach_ids)

#dates = ['20250308', '20250324', # we got these
dates = ['20250327']  # we didn't get this one?
#dates =  ['20250815', '20250826', '20250829', '20250831', '20250912', '20250913', '20250914', '20251010',  '20251011', '20251016']
#dates = ['20250306']
hours = np.arange(1,5*24+1,1)

for date_str in dates:
    file_name_out = '/dataSIO/PFM_Simulations/Archive/riverQ_from_matt/LV4_riverQ_' + date_str + '.nc'
    print('making ', file_name_out)
    cnt = 0
    t3 = []
    q3 = np.zeros((len(hours),len(reach_ids)))
    for hr in hours:
        fore_hr_str = str(hr).zfill(3)
        url = 'https://storage.googleapis.com/national-water-model/nwm.' + \
        date_str + \
        '/medium_range_blend/nwm.t00z.medium_range_blend.channel_rt.f' + \
        fore_hr_str + \
        '.conus.nc#mode=bytes'

        with nc.Dataset(url) as ds:
            id = ds.variables['feature_id'][:]
            t =  ds.variables['time']
            q = ds.variables
            date2 = nc.num2date(t[:],t.units).data
            std_dt = datetime.strptime(date2[0].isoformat(), "%Y-%m-%dT%H:%M:%S")
            t3.append( std_dt )
            
            mask_val1 = (id == reach_ids[0])
            mask_val2 = (id == reach_ids[1])
            mask_val3 = (id == reach_ids[2])

            # Combine the masks using logical OR
            combined_mask = mask_val1 | mask_val2 | mask_val3

            # Get the indices where the combined mask is True
            ig = np.where(combined_mask)
            # the ordering of ig is: TJ, Otay, SW - it got reversed 

            qq = ds.variables['streamflow'][ig]

            q3[cnt,:] = qq

            reach_ids_2 = id[ig]
            cnt = cnt + 1

    t4 = np.array(t3)
    print('calling make_Qnc')
    make_Qnc(t4,q3,reach_ids_2,file_name_out)




making  /dataSIO/PFM_Simulations/Archive/riverQ_from_matt/LV4_riverQ_20250327.nc


curlcode: (56)Failure when receiving data from the peer : Proxy CONNECT aborted


OSError: [Errno -101] NetCDF: HDF error: 'https://storage.googleapis.com/national-water-model/nwm.20250327/medium_range_blend/nwm.t00z.medium_range_blend.channel_rt.f078.conus.nc#mode=bytes'

In [11]:
dsr2 = nc.Dataset('/dataSIO/PFM_Simulations/Archive/riverQ_from_matt/LV4_riverQ_20250306.nc')
dsr1 = nc.Dataset('/dataSIO/PFM_Simulations/Archive/riverQ_from_matt/LV4_riverQ_20250228.nc')
print(dsr1)
print(dsr2)
print(dsr1.variables['station_id'][:])
print(dsr2.variables['station_id'][:])
#print(dsr2.variables['discharge'][:,0])


<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    title: discharge for the 3 LV4 rivers
    history: Created on 2025-07-12 11:33:36
    dimensions(sizes): time(120), station(3)
    variables(dimensions): float64 time(time), float32 discharge(time, station), <class 'str'> station_id(station)
    groups: 
<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    title: discharge from National Water Model: medium range blend forecast
    history: Created on 2025-10-23 19:45:40
    dimensions(sizes): time(120), station(3)
    variables(dimensions): float64 time(time), float64 discharge(time, station), <class 'str'> station_id(station), int64 reach_id(station)
    groups: 
['Sweewater' 'Otay' 'TJRE']
['Sweewater' 'Otay' 'TJRE']
